In [1]:
from pystac_client import Client
#get catalog
catalog = Client.open("https://montandon-eoapi-stage.ifrc.org/stac/")#"https://montandon-eoapi-stage.ifrc.org/stac/api")


In [2]:
#print collections
collections = list(catalog.get_collections())
collections

[<CollectionClient id=desinventar-events>,
 <CollectionClient id=desinventar-impacts>,
 <CollectionClient id=emdat-events>,
 <CollectionClient id=emdat-hazards>,
 <CollectionClient id=emdat-impacts>,
 <CollectionClient id=gdacs-events>,
 <CollectionClient id=gdacs-hazards>,
 <CollectionClient id=gdacs-impacts>,
 <CollectionClient id=gfd-events>,
 <CollectionClient id=gfd-hazards>,
 <CollectionClient id=gfd-impacts>,
 <CollectionClient id=glide-events>,
 <CollectionClient id=glide-hazards>,
 <CollectionClient id=ibtracs-events>,
 <CollectionClient id=ibtracs-hazards>,
 <CollectionClient id=idmc-gidd-events>,
 <CollectionClient id=idmc-gidd-impacts>,
 <CollectionClient id=idmc-idu-events>,
 <CollectionClient id=idmc-idu-impacts>,
 <CollectionClient id=ifrcevent-events>,
 <CollectionClient id=ifrcevent-hazards>,
 <CollectionClient id=ifrcevent-impacts>,
 <CollectionClient id=pdc-events>,
 <CollectionClient id=pdc-hazards>,
 <CollectionClient id=pdc-impacts>,
 <CollectionClient id=referenc

In [3]:
# try a specific query
#ifrc_impact_collection = catalog.get_collection("ifrcevent-events"
from climada.util.coordinates import get_country_geometries
ctry_bbox = get_country_geometries("SDN").total_bounds
event_dates = ['2024-07-01T00:00:00Z', '2024-10-02T00:00:00Z']
results = catalog.search(
    collections=["ifrcevent-impacts", "ifrcevent-hazards", "ifrcevent-events"],
    bbox=ctry_bbox,
    datetime=event_dates
    #query={"limit": 1},

)


In [4]:
for item in results.items():
    print(item.properties)

{'roles': ['source', 'event'], 'title': 'Chad - Floods 2024', 'datetime': '2024-09-16T08:24:34Z', 'description': '<p style="padding-left: 80px;"><span class="NormalTextRun SCXW227376754 BCX0"><span class="SCXW232751398 BCX0"><span class="WACImageContainer NoPadding DragDrop BlobObject SCXW232751398 BCX0" role="presentation"><img class="WACImage SCXW232751398 BCX0" src="https://prddsgofilestorage.blob.core.windows.net/api/images/7144e.jpg" width="1021" height="574"></span></span></span></p>\r\n<p><span class="NormalTextRun SCXW227376754 BCX0">Weeks of\xa0</span><span class="NormalTextRun SCXW227376754 BCX0">severe rains in Chad have</span><span class="NormalTextRun SCXW227376754 BCX0"> hit all 23 provinces leaving</span><span class="NormalTextRun SCXW227376754 BCX0"> at least 340 people dead</span><span class="NormalTextRun SCXW227376754 BCX0"> and 1.5 million affected since July. </span><span class="NormalTextRun SCXW227376754 BCX0">The situation continues to evolve very rapidly</span>

In [5]:
#ifrc impacts collections
item_collection_all_ifrc = catalog.search(
    collections=["ifrcevent-impacts","ifrcevent-hazards", "ifrcevent-events"],#"ifrcevent-impacts"
).item_collection()

In [6]:
#item_collection_all_imp = catalog.search(
#    collections=["ifrcevent-impacts", "desinventar-impacts", "emdat-impacts",
#                 "gdacs-impacts", "gfd-impacts", "idmc-gidd-impacts","idmc-idu-impacts",
#                 "pdc-impacts", "usgs-impacts"],
#).item_collection()

In [7]:
#try loading collections into xarray using stackstac
#fail due to missing "assets" field in collections?
#import stackstac

#stack_ifrc = stackstac.stack(item_collection_all_ifrc , assets=None)

In [8]:
# try using geopandas (https://notebooksharing.space/view/fb24fa1b56d1499b88da3df2808b9ba567c33b01f8635a8e308087556b2991fc#displayOptions=)
import pandas as pd
import geopandas
import numpy as np

def lists_to_str(x):
    if isinstance(x, list):
        return ", ".join(x)
    else:
        return x

def from_items(item_collection):
    gdf = geopandas.GeoDataFrame.from_features(item_collection.to_dict()["features"]).set_crs(4326)
    gdf["datetime"] = pd.to_datetime(gdf.datetime, format="mixed")
    gdf["id"] = [x.id for x in item_collection]
    gdf = gdf.map(lambda x: lists_to_str(x)) # remove lists
    gdf = make_impact_cols(gdf)
    return gdf

def make_impact_cols(gdf, col_in="monty:impact_detail", cols_out=["type", "value", "category", "estimate_type"]):
    for col in cols_out:
        gdf["impact_"+col] = gdf[col_in].apply(lambda x: x.get(col, None) if isinstance(x, dict) else np.nan)
    gdf = gdf.drop(col_in, axis=1)
    return gdf

gdf = from_items(item_collection_all_ifrc)

In [9]:
gdf.impact_category.value_counts()

impact_category
people    1312
Name: count, dtype: int64

In [10]:
gdf.impact_type.value_counts().keys()

Index(['affected_total', 'death', 'assisted', 'displaced_total', 'injured',
       'missing', 'potentially_affected', 'highest_risk'],
      dtype='object', name='impact_type')

In [11]:
gdf.impact_estimate_type.value_counts()

impact_estimate_type
primary    1312
Name: count, dtype: int64

In [12]:
gdf.roles.value_counts()

roles
source, event     2177
source, impact    1312
Name: count, dtype: int64

In [13]:
gdf.columns

Index(['geometry', 'roles', 'title', 'datetime', 'description', 'end_datetime',
       'monty:etl_id', 'monty:corr_id', 'start_datetime', 'monty:hazard_codes',
       'monty:country_codes', 'monty:episode_number', 'id', 'impact_type',
       'impact_value', 'impact_category', 'impact_estimate_type'],
      dtype='object')

In [14]:
gdf["monty:etl_id"]

0       8af74cfb-ddd6-4b18-b097-7436487383e6
1       86516c7f-b238-483e-b29c-e38a8fa7550d
2       5c0a04bd-a447-40d2-9f85-ae5ca1414abd
3       4130c5c4-e82d-4a4f-9241-c403a908bba2
4       ba67afe9-1976-446a-889e-e4bf013c0e97
                        ...                 
3484    d953288c-7be7-40e2-902b-7921114b5534
3485    f0f14355-8108-47ee-b24e-67fab621f7df
3486    3677b72d-e7d5-4f54-bf5c-749880471467
3487    f08b2186-bc57-4aa4-9690-890143019feb
3488    c2021217-f9be-4eca-90ae-692c4906e73d
Name: monty:etl_id, Length: 3489, dtype: object

In [15]:
gdf.columns

Index(['geometry', 'roles', 'title', 'datetime', 'description', 'end_datetime',
       'monty:etl_id', 'monty:corr_id', 'start_datetime', 'monty:hazard_codes',
       'monty:country_codes', 'monty:episode_number', 'id', 'impact_type',
       'impact_value', 'impact_category', 'impact_estimate_type'],
      dtype='object')